# Create SESSIONS dataframe 

In [ ]:
# 1. File Path - CSV Workflow

# Use clean CSV not parquet RELATIVE path with repo folder structure! 
# Assuming column names in order are: (client_id, visitor_id, visit_id, process_step, date_time)

import pandas as pd
import numpy as np

# File paths using relative routing from the /notebooks folder -- INSERT PATH TODO
RAW_WEB_DATA_PATH = '../data/raw/  FILENAME .csv'
CLEAN_WEB_DATA_PATH = '../data/clean/     FILENAME   .csv'

# Load the merged CSV
df_web = pd.read_csv(RAW_WEB_DATA_PATH)
df_web['date_time'] = pd.to_datetime(df_web['date_time'])

In [ ]:
# 2. Isolating Journeys and Sessions

# A Session is defined by a unique visit_id
# A Journey is the chronological sequence of process_steps within that session. 
# Because users can go backward, sort by time BEFORE grouping

# Sort chronologically to unravel the timeline
df_web = df_web.sort_values(by=['client_id', 'visit_id', 'date_time'], ascending=[True, True, True])

# Extract Session Journeys (List of steps)
journeys = df_web.groupby(['client_id', 'visit_id'])['process_step'].apply(list).reset_index(name='step_sequence')

# Calculate Session Metrics (Time in funnel & Step count)
session_times = df_web.groupby(['client_id', 'visit_id']).agg(
    start_time=('date_time', 'min'),
    end_time=('date_time', 'max'),
    total_steps=('process_step', 'count')
).reset_index()

# Calculate total time per session in seconds (END time minus START time)
session_times['time_in_funnel_sec'] = (session_times['end_time'] - session_times['start_time']).dt.total_seconds()


In [ ]:
# Merge back together into a clean "Session-Level" DataFrame
df_sessions = pd.merge(journeys, session_times, on=['client_id', 'visit_id'])

# Save to clean data folder
df_sessions.to_csv(CLEAN_WEB_DATA_PATH, index=False)

## KPIs & Metrics Analysis - Compare with L's notes, Separate Error rate and Dropoff rate more clearly TODO

Metrics are raw, quantifiable data points (see Self guided KPI lesson). They belong in the df_sessions DataFrame as new columns.

KPIs (Key Performance Indicators) are strategic metrics tied directly to the business goal of the A/B test. 
Calculate these ACROSS the Test vs. Control groups, not per user!

## List of Metrics & KPIs we need (from Trello) -- "Feature Engineering" 

total_steps: 
Already calculated above!

reached_confirm (Boolean): 
Extracted from step_sequence. Did the list contain the last 'confirm' stage? TODO

time_in_funnel: 
Already calculated above (time_in_funnel_sec)

is_error_sequence (Boolean/Count): 
Extracted by looking for backward movement in step_sequence (e.g., Step 3 followed by Step 2) TODO

sessions_per_client: Count of visit_ids per client_id TODO

## Core KPIs (For Statistical Testing between Test vs. Control):

Primary KPI: Completion Rate
Calculation: Sum of clients w. (reached_confirm) v. Total clients in group

Secondary KPI 1: Time to Completion
Calculation: Average time_in_funnel_sec (Filtered only for successful sessions)

Secondary KPI 2: Error / Friction Rate --> SEPARATE CLEARLY, DISCUSS TODO
Calculation: % of sessions containing backward steps or dropping off before confirm.

Secondary KPI 3: Drop-off Rate per Step ("Step Conversion Rate")
Calculation: % of users who exit at step 1, step 2, etc. (Useful for "funnel" visualization)

Secondary KPI 4: "Call rate" vs Online completion rates - fewer calls indicate better online completion? --> (OPTIONAL, IF TIME)   


# Next Steps for web_data functions before Hypo Testing (AI Tutor) 

Translate the "Metrics" list into Python code, applied to df_sessions -- the actual H0/H1 testing! 

Flag Completions: Add a column 'completed' to df_sessions, by checking if 'confirm' is in each user's step_sequence list

Flag Errors: Write a function to detect if the step_sequence list breaks chronological order 
Example Journey (['start', 'step_1', 'step_2', 'step_1']) --> Highlight any BACKSTEPS in a user journey

Aggregate to Client Level: Group df_sessions by client_id --> Get Total time spent across all sessions, total errors, and total success / "completed" status

Master Merge: Once we aggregate to ONE ROW per client_id, join it with the DE-ANONYMIZED df_clients (Demographics + Roster) --  pd.merge() or agg method? Use that master file for all our A/B test hypothesis coding 